# BPR (Bayesian Personalized Ranking)

## Теорема Байеса

В основе баесовских методов лежит формула Байеса:

$$
P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}
$$

Где:  
- $P(A)$ — априорная вероятность события $A$
- $P(B)$ — априорная вероятность события $B$  
- $P(B \mid A)$ — вероятность события $B$ при условии $A$  
- $P(A \mid B)$ — апостериорная вероятность события $A$ при наличии $B$  

## BPR (Bayesian Personalized Ranking)

BPR (Bayesian Personalized Ranking) — это метод коллаборативной фильтрации, разработанный специально для рекомендаций с неявными данными, такими как клики, просмотры или лайки. В отличие от стандартных подходов, основанных на явных оценках (рейтингах), BPR лучше справляется с ситуациями, когда данные не содержат явных оценок.

### Идея BPR:
Основная идея BPR заключается в следующем:

- У пользователя больше интерес к элементам, с которыми он взаимодействовал (например, прочитал новость), чем к тем, с которыми не взаимодействовал.
- Задача метода — ранжировать взаимодействованные объекты выше невзаимодействованных.

### Пример:
Если пользователь **U** прочитал новость **A**, но не читал новость **B**, модель должна предсказать, что **A** предпочтительнее **B** для пользователя **U**.

### Формулировка задачи:
Пусть **U** — множество пользователей, **I** — множество новостей.  
Для пользователя **u** существует пара взаимодействий (положительное — **i**, отрицательное — **j**).  
- Положительные взаимодействия — новости, которые пользователь прочитал.
- Отрицательные взаимодействия — случайные новости, которые пользователь не читал.  

Цель: максимизировать вероятность того, что пользователь предпочтет **i** над **j**.

### Функция вероятности и ранжирования:
Для каждой тройки (**u**, **i**, **j**), где:

- **u** — пользователь,
- **i** — положительное взаимодействие (прочитанная новость),
- **j** — отрицательное взаимодействие (непрочитанная новость),

максимизируем вероятность:

$$
P(i >_u j) = \sigma(\hat{x}_{ui} - \hat{x}_{uj})
$$

где:

- $\sigma$ — сигмоида: $$\sigma(x) = \frac{1}{1 + e^{-x}}$$
- $\hat{x}_{ui}$ — предсказанная "полезность" объекта **i** для пользователя **u**.
- $\hat{x}_{uj}$ — предсказанная "полезность" объекта **j** для пользователя **u**.

### Функция потерь:
Оптимизация происходит через максимизацию логарифма правдоподобия:

$$
L = \sum_{(u, i, j) \in D} \log(\sigma(\hat{x}_{ui} - \hat{x}_{uj})) - \lambda \| \Theta \|_2^2
$$

где:

- **D** — множество всех возможных тройных комбинаций (**u**, **i**, **j**).
- **λ** — коэффициент регуляризации для контроля переобучения.
- **Θ** — параметры модели (матрицы эмбеддингов пользователей и новостей).

### Реализация модели:
Обычно BPR используется в сочетании с матричной факторизацией. Эмбеддинги пользователей и новостей обучаются одновременно:

$$
\hat{x}_{ui} = \langle p_u, q_i \rangle = \sum_{f=1}^F p_{uf} \cdot q_{if}
$$

где:

- $p_u$ — вектор эмбеддинга пользователя **u**.
- $q_i$ — вектор эмбеддинга новости **i**.
- **F** — размерность скрытого пространства.

### Пошаговый алгоритм BPR:
1. **Инициализация**: случайная инициализация эмбеддингов пользователей и новостей.
2. **Формирование триплетов** (**u**, **i**, **j**): Для каждого пользователя **u** выбирается положительная новость **i**. Случайным образом выбирается отрицательная новость **j**.
3. **Обновление эмбеддингов**: используется стохастический градиентный спуск (SGD) для обновления эмбеддингов.
4. **Оптимизация**: максимизация правдоподобия или минимизация функции потерь.
5. **Рекомендации**: на основе полученных эмбеддингов строится ранжированный список рекомендаций.


Пробная реализация на небольших данных

In [ ]:
!pip install numpy
!pip install pandas

Загрузка первых 1000 строк таблицы behaviors (поведение пользователей - показанные/просмотренные новости).

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../data/Mindsmall_train/behaviors.tsv', nrows=1000, sep='\t', header=None, names=['id', 'user_id', 'timestamp', 'history', 'clicked_news'])

print(df.head(5))

   id user_id              timestamp  \
0   1  U13740  11/11/2019 9:05:58 AM   
1   2  U91836  11/12/2019 6:11:30 PM   
2   3  U73700  11/14/2019 7:01:48 AM   
3   4  U34670  11/11/2019 5:28:05 AM   
4   5   U8125  11/12/2019 4:11:21 PM   

                                             history  \
0  N55189 N42782 N34694 N45794 N18445 N63302 N104...   
1  N31739 N6072 N63045 N23979 N35656 N43353 N8129...   
2  N10732 N25792 N7563 N21087 N41087 N5445 N60384...   
3  N45729 N2203 N871 N53880 N41375 N43142 N33013 ...   
4                        N10078 N56514 N14904 N33740   

                                        clicked_news  
0                                  N55689-1 N35729-0  
1  N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...  
2  N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...  
3                N35729-0 N33632-0 N49685-1 N27581-0  
4  N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...  


Обратить внимание нужно на последний столбец, в нем содержится информация о том, какие новости были предложены, и на какие из них пользователь кликнул (1 - клик, 0 - нет).

Например, пользователю с id 1 были предложены 2 новости - N55689 и N35729. На первую он кликнул (N55689-1), а вторую проигнорировал (N35729-0). Предполагаем, что те новости, которые остались проигнорированы, менее интересны. Таким образом, положительной новостью будет считаться та, на которую пользователь кликнул, а отрицательной - проигнорированная.

Далее - **формирование триплетов**. Каждая строка таблицы - момент времени, в который пользователь кликнул на одну из предложенных новостей. Триплеты будут использоваться для обучения модели, поэтому для каждого клика их нужно составить как можно больше. Если было предложено $N$ новостей, то можно составить, как минимум, $N-1$ триплетов, используя только реально предложенные варианты.

In [2]:
# Функция для генерации триплетов (user, pos, neg)

def generate_triplets(df):
    triplets = []

    for _, row in df.iterrows():
        user = row['user_id']
        impressions = row['clicked_news'].split() # разделение списка предложенных новостей

        # разделение списка impressions по факту клика на новость
        clicked = [news[:-2] for news in impressions if news[-1] == '1']
        not_clicked = [news[:-2] for news in impressions if news[-1] == '0']

        # создание триплетов (n - 1 штук для каждого клика)
        for pos in clicked:
            for neg in not_clicked:
                triplets.append((user, pos, neg))

    return np.array(triplets)
    
triplets = generate_triplets(df)
print(triplets[:5])

[['U13740' 'N55689' 'N35729']
 ['U91836' 'N17059' 'N20678']
 ['U91836' 'N17059' 'N39317']
 ['U91836' 'N17059' 'N58114']
 ['U91836' 'N17059' 'N20495']]


Итак, триплеты получены. Теперь нужно **обучить модель BPR**: нужно оптимизировать параметры модели таким образом, чтобы она училась оценивать релевантность новостей для каждого пользователя на основе этих триплетов.

**Модель BPR** состоит из:

- **Предсказания релевантности** новостей (с использованием латентных векторов)
- **Минимизации функции потерь**, которая основана на разнице между положительными и отрицательными новостями

В итоге будут построены **латентные векторы** - векторы, состоящие из неявных параметров и представляющие предпочтения пользователей или характеристики новостей. Допустим, у нас есть несколько пользователей и несколько новостей. Каждому пользователю соответствует латентный вектор, который будет характеризовать его интересы. Каждой новости тоже соответствует свой вектор, который будет описывать её содержание или категорию.

**Простой пример:**

**Пользователь U1:**

Латентный вектор:
$p_{U1} = [0.2, 0.4, 0.7]$

**Новость N1:**

Латентный вектор:
$q_{N1} = [0.1, 0.5, 0.8]$

**Новость N2:**

Латентный вектор:
$q_{N2} = [0.3, 0.6, 0.9]$

Чтобы предсказать, насколько интересна новость $N1$ для пользователя $U1$, мы рассчитываем скалярное произведение (или внутреннее произведение) между векторами $p_{U1}$ и $q_{N1}$:

$$
p_{U1} \cdot q_{N1} = 0.2 \times 0.1 + 0.4 \times 0.5 + 0.7 \times 0.8 = 0.59
$$

Если это значение высокое, то мы говорим, что пользователь заинтересован в новости $N1$.

In [3]:
# заменим строковые представления пользователей и новостей на числовые индексы

unique_users = np.unique(triplets[:, 0]) # уникальные пользователи
unique_items = np.unique(np.concatenate([triplets[:, 1], triplets[:, 2]])) # все уникальные новости

# создание отображения строковых ID -> числовые индексы

user_ids = {u: idx for idx, u in enumerate(unique_users)}
item_ids = {i: idx for idx, i in enumerate(unique_items)}

# преобразование данных в числовой формат
triplets_idx = np.array([[user_ids[u], item_ids[i], item_ids[j]] for u, i, j in triplets], dtype=np.int32)

print(triplets_idx[:5])

[[  56 3186 1760]
 [ 946  472  709]
 [ 946  472 2044]
 [ 946  472 3365]
 [ 946  472  698]]


BPR использует **матрицы эмбеддингов пользователей и новостей**, каждый пользователь и новость представляется вектором размерности k.

In [10]:
# гиперпараметры модели
k = 10 # размерность эмбеддинга
lr = 0.01 # скорость обучения
reg = 0.001 # регуляризация
epochs = 100 # количество эпох обучения

# инициализация случайных эмбеддингов (нормальное распределение)
P = np.random.normal(0, 0.1, (len(user_ids), k)) # эмбеддинги пользователей
Q = np.random.normal(0, 0.1, (len(item_ids), k)) # эмбеддинги новостей

Теперь `P` и `Q` - это **матрицы эмбеддингов** размером:

- `P.shape = (len(user_ids), k)`
- `Q.shape = (len(item_ids), k)`

Каждая строка в `P` - это вектор пользователя \
Каждая строка в `Q` - это вектор новости

## Сигмоида

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

## Градиентный спуск

In [12]:
# Обучение модели
for epoch in range(epochs):
    total_loss = 0  # Переменная для хранения функции потерь
    
    # Перебираем триплеты (u, i, j)
    for u, i, j in triplets_idx:
        # Вычисляем разницу между позитивной и негативной новостью
        x_uij = np.dot(P[u], Q[i]) - np.dot(P[u], Q[j])
        sigmoid_x = sigmoid(x_uij)
        
        # Градиенты
        grad_Pu = (1 - sigmoid_x) * (Q[i] - Q[j]) - reg * P[u]
        grad_Qi = (1 - sigmoid_x) * P[u] - reg * Q[i]
        grad_Qj = -(1 - sigmoid_x) * P[u] - reg * Q[j]

        # Обновляем эмбеддинги
        P[u] += lr * grad_Pu
        Q[i] += lr * grad_Qi
        Q[j] += lr * grad_Qj

        # Функция потерь (без регуляризации)
        total_loss += np.log(sigmoid_x)

    print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: -227.0380
Epoch 2, Loss: -225.1241
Epoch 3, Loss: -223.2595
Epoch 4, Loss: -221.4424
Epoch 5, Loss: -219.6712
Epoch 6, Loss: -217.9442
Epoch 7, Loss: -216.2600
Epoch 8, Loss: -214.6171
Epoch 9, Loss: -213.0141
Epoch 10, Loss: -211.4496
Epoch 11, Loss: -209.9224
Epoch 12, Loss: -208.4312
Epoch 13, Loss: -206.9749
Epoch 14, Loss: -205.5523
Epoch 15, Loss: -204.1624
Epoch 16, Loss: -202.8041
Epoch 17, Loss: -201.4764
Epoch 18, Loss: -200.1784
Epoch 19, Loss: -198.9092
Epoch 20, Loss: -197.6678
Epoch 21, Loss: -196.4535
Epoch 22, Loss: -195.2654
Epoch 23, Loss: -194.1027
Epoch 24, Loss: -192.9647
Epoch 25, Loss: -191.8506
Epoch 26, Loss: -190.7598
Epoch 27, Loss: -189.6916
Epoch 28, Loss: -188.6453
Epoch 29, Loss: -187.6203
Epoch 30, Loss: -186.6161
Epoch 31, Loss: -185.6319
Epoch 32, Loss: -184.6674
Epoch 33, Loss: -183.7219
Epoch 34, Loss: -182.7949
Epoch 35, Loss: -181.8859
Epoch 36, Loss: -180.9944
Epoch 37, Loss: -180.1199
Epoch 38, Loss: -179.2621
Epoch 39, Loss: -178.

In [14]:
def recommend(user_str, top_n=5):
    user_idx = user_ids.get(user_str, None)
    if user_idx is None:
        print("Пользователь не найден")
        return []
    
    user_vector = P[user_idx]  # Вектор пользователя

    # Считаем предсказанные рейтинги для всех новостей
    scores = np.dot(Q, user_vector)

    # Получаем топ-N индексов
    top_items = np.argsort(scores)[::-1][:top_n]

    # Преобразуем обратно в строковые ID
    recommended_news = [news for news, idx in item_ids.items() if idx in top_items]
    
    return recommended_news

# Пример рекомендации для пользователя "U12345"
print("Рекомендованные новости:", recommend("U13740"))

Рекомендованные новости: [np.str_('N15462'), np.str_('N50107'), np.str_('N51006'), np.str_('N55689'), np.str_('N56211')]
